In [2]:
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")

!git clone https://{token}@github.com/eslamfawzy72/from-pixels-to-language.git
%cd /content/from-pixels-to-language
!git remote set-url origin https://github.com/eslamfawzy72/from-pixels-to-language.git
!git log --oneline -1
!pip install -q -r requirements.txt

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")


Cloning into 'from-pixels-to-language'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 127 (delta 38), reused 107 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 12.79 MiB | 9.83 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/from-pixels-to-language
4ca5664 (HEAD -> main, origin/main, origin/HEAD, origin/CNN-from-scratch) merged main with cnn from scratch branch
torch 2.11.0+cu130 | CUDA: True | Tesla T4


In [3]:
!python -m unittest -v 2>&1 | tail -25


test_seed_makes_initialization_reproducible (tests.test_training.TestCheckpointAndSeed.test_seed_makes_initialization_reproducible) ... ok
test_accuracy (tests.test_training.TestLosses.test_accuracy) ... ok
test_aux_weight_applied (tests.test_training.TestLosses.test_aux_weight_applied) ... ok
test_split_outputs (tests.test_training.TestLosses.test_split_outputs) ... ok
test_best_epoch_uses_monitor_direction (tests.test_training.TestTrainingLoop.test_best_epoch_uses_monitor_direction) ... ok
test_fit_writes_checkpoints_and_history (tests.test_training.TestTrainingLoop.test_fit_writes_checkpoints_and_history) ... ok
test_googlenet_aux_outputs_in_training_loop (tests.test_training.TestTrainingLoop.test_googlenet_aux_outputs_in_training_loop) ... ok
test_invalid_config (tests.test_training.TestTrainingLoop.test_invalid_config) ... ok
test_train_and_validate_return_metrics (tests.test_training.TestTrainingLoop.test_train_and_validate_return_metrics) ... ok

--------------------------------

In [4]:
!python -m src.data.dataset


README.md: 100% 7.16k/7.16k [00:00<00:00, 17.8MB/s]

data/train-00000-of-00002-5e7d350dbc117c(…): downloading bytes:  64% 184M/287M [00:01<00:00, 235MB/s, 10.8MB/s  ]
data/train-00000-of-00002-5e7d350dbc117c(…): downloading bytes:  93% 267M/287M [00:01<00:00, 295MB/s, 21.2MB/s  ]
data/train-00000-of-00002-5e7d350dbc117c(…): downloading bytes: 100% 285M/285M [00:02<00:00, 117MB/s, 25.9MB/s  ]
data/train-00000-of-00002-5e7d350dbc117c(…): reconstructing file: 100% 287M/287M [00:02<00:00, 118MB/s, 26.8MB/s  ]

data/train-00001-of-00002-ae08e95fea856c(…): downloading bytes:  89% 265M/297M [00:01<00:00, 266MB/s, 19.0MB/s  ]
data/train-00001-of-00002-ae08e95fea856c(…): reconstructing file:  23% 67.1M/297M [00:02<00:06, 33.0MB/s]
data/train-00001-of-00002-ae08e95fea856c(…): downloading bytes: 100% 295M/295M [00:02<00:00, 111MB/s, 27.3MB/s  ]
data/train-00001-of-00002-ae08e95fea856c(…): reconstructing file: 100% 297M/297M [00:02<00:00, 111MB/s, 27.7MB/s  ]
Generating train split: 100% 5994/5994

In [5]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ROOT = "/content/drive/MyDrive/from-pixels-to-language/runs"
print("Runs will be saved to:", RUN_ROOT)


Mounted at /content/drive
Runs will be saved to: /content/drive/MyDrive/from-pixels-to-language/runs


In [6]:
from src.data.dataset import get_dataloaders
from src.models import create_model, count_parameters
from src.training import TrainConfig, fit, set_seed

train_loader, val_loader, test_loader = get_dataloaders(batch_size=32, num_workers=2)

config = TrainConfig(model_name="resnet18", num_classes=200, epochs=3,
                     optimizer="sgd", learning_rate=0.01, batch_size=32)
set_seed(config.seed)
model = create_model(config.model_name, num_classes=config.num_classes)
print("parameters:", count_parameters(model)[0])

def show(epoch, m):
    print(f"epoch {epoch}: train_loss={m['train_loss']:.3f} train_acc={m['train_accuracy']:.3f} "
          f"val_loss={m['val_loss']:.3f} val_acc={m['val_accuracy']:.3f} ({m['epoch_seconds']:.0f}s)")

result = fit(model, train_loader, val_loader, config,
             run_dir=f"{RUN_ROOT}/resnet18_sanity", on_epoch_end=show)
print("best epoch:", result["best_epoch"], "| best val acc:", round(result["best_metric"], 4))


parameters: 11279112
epoch 1: train_loss=5.424 train_acc=0.004 val_loss=5.204 val_acc=0.010 (65s)
epoch 2: train_loss=5.133 train_acc=0.012 val_loss=5.037 val_acc=0.020 (60s)
epoch 3: train_loss=4.986 train_acc=0.021 val_loss=5.022 val_acc=0.021 (64s)
best epoch: 3 | best val acc: 0.0211


In [7]:
!python -m unittest tests.test_cli tests.test_training 2>&1 | tail -3


}
alexnet    params=  61,100,840  latency=    22.5 ±    0.7 ms  peak_mem=n/a
Saved /tmp/tmpyxxsrgjp/bench.json and /tmp/tmpyxxsrgjp/bench.csv


In [9]:
!python -m unittest tests.test_cli tests.test_training 2>&1


E{
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "processor": "x86_64",
  "cpu_threads": 1,
  "device": "cpu",
  "cuda_available": true
}
alexnet    params=  61,100,840  latency=    25.6 ±    3.1 ms  peak_mem=n/a
Saved /tmp/tmpnc0mo6pl/bench.json and /tmp/tmpnc0mo6pl/bench.csv
............
ERROR: test_cli (unittest.loader._FailedTest.test_cli)
----------------------------------------------------------------------
ImportError: Failed to import test module: test_cli
Traceback (most recent call last):
  File "/usr/lib/python3.13/unittest/loader.py", line 141, in loadTestsFromName
    module = __import__(module_name)
ModuleNotFoundError: No module named 'tests.test_cli'


----------------------------------------------------------------------
Ran 13 tests in 7.384s

FAILED (errors=1)


In [10]:
!python -m unittest tests.test_cli tests.test_training 2>&1 | grep -E "^(Ran|OK|FAILED|ERROR|FAIL)"


ERROR: test_cli (unittest.loader._FailedTest.test_cli)
Ran 13 tests in 7.122s
FAILED (errors=1)


In [11]:
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
!git pull https://{token}@github.com/eslamfawzy72/from-pixels-to-language.git main
!pip install -q -r requirements.txt
!git log --oneline -1


remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 33 (delta 17), reused 33 (delta 17), pack-reused 0 (from 0)
Unpacking objects: 100% (33/33), 19.87 KiB | 1.81 MiB/s, done.
From https://github.com/eslamfawzy72/from-pixels-to-language
 * branch            main       -> FETCH_HEAD
Updating 4ca5664..f32f0cd
Fast-forward
 .gitignore                  |   3 +
 README.md                   |  73 +++++++++++++++--
 configs/classification.yaml |  37 +++++++++
 requirements.txt            |   4 +
 src/cli.py                  | 184 ++++++++++++++++++++++++++++++++++++++++++
 src/tasks/__init__.py       |   1 +
 src/tasks/classification.py | 142 ++++++++++++++++++++++++++++++++
 src/training/__init__.py    |   3 +-
 src/training/checkpoint.py  |   6 +-
 src/training/config.py      |  44 +++++++++-
 src/training/tracking.py    | 117 +++++++++++++++++++++++++++
 src/training/train.py       |  82 ++++++++

In [12]:
!pip install -q -r requirements.txt
!python -m unittest tests.test_cli tests.test_training 2>&1 | grep -E "^(Ran|OK|FAILED|ERROR|FAIL)"


Ran 26 tests in 37.208s
OK


In [13]:
from google.colab import drive
drive.mount('/content/drive')          # skip if already mounted

DRIVE = "/content/drive/MyDrive/from-pixels-to-language"
!python -m src.cli train --task classification --epochs 3 --run-name cli_sanity \
    --set train.warmup_epochs=1 \
    --output-dir {DRIVE}/runs --tracking-dir {DRIVE}/mlflow


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2026/10/30 14:00:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/30 14:00:28 INFO mlflow.store.db.utils: Updating database tables
Training resnet18: 11,279,112 parameters, 5091 train / 899 val images -> /content/drive/MyDrive/from-pixels-to-language/runs/cli_sanity
epoch   1/3  lr=0.02500  train_loss=5.449 train_acc=0.003  val_loss=5.317 val_acc=0.006  (64s)
epoch   2/3  lr=0.05000  train_loss=5.281 train_acc=0.006  val_loss=5.267 val_acc=0.008  (61s)
epoch   3/3  lr=0.02500  train_loss=5.172 train_acc=0.010  val_loss=5.124 val_acc=0.019  (64s)
Best val_accuracy: 0.0189 at epoch 3


In [14]:
!python -m src.cli train --task classification --run-name resnet18_baseline \
    --output-dir {DRIVE}/runs --tracking-dir {DRIVE}/mlflow


Training resnet18: 11,279,112 parameters, 5091 train / 899 val images -> /content/drive/MyDrive/from-pixels-to-language/runs/resnet18_baseline
epoch   1/60  lr=0.01250  train_loss=5.415 train_acc=0.005  val_loss=5.225 val_acc=0.010  (64s)
epoch   2/60  lr=0.02500  train_loss=5.225 train_acc=0.009  val_loss=5.109 val_acc=0.012  (61s)
epoch   3/60  lr=0.03750  train_loss=5.138 train_acc=0.015  val_loss=5.063 val_acc=0.013  (64s)
epoch   4/60  lr=0.05000  train_loss=5.048 train_acc=0.019  val_loss=4.943 val_acc=0.021  (64s)
epoch   5/60  lr=0.04996  train_loss=4.923 train_acc=0.025  val_loss=4.784 val_acc=0.040  (64s)
epoch   6/60  lr=0.04985  train_loss=4.772 train_acc=0.040  val_loss=4.870 val_acc=0.034  (64s)
epoch   7/60  lr=0.04966  train_loss=4.630 train_acc=0.052  val_loss=4.408 val_acc=0.059  (62s)
epoch   8/60  lr=0.04939  train_loss=4.515 train_acc=0.060  val_loss=4.566 val_acc=0.061  (64s)
epoch   9/60  lr=0.04906  train_loss=4.416 train_acc=0.074  val_loss=4.225 val_acc=0.079 

In [15]:
!python -m src.cli evaluate --task classification --checkpoint {DRIVE}/runs/resnet18_baseline/best.pt --split test


test: loss=2.3696 accuracy=0.4584 on 5794 images -> /content/drive/MyDrive/from-pixels-to-language/runs/resnet18_baseline/test_metrics.json
